# 🚀 YOLOv11 Entrenamiento MEJORADO en Google Colab con GPU

## 🎯 MEJORAS vs versión anterior:
- ✅ **100 épocas** (vs 50)
- ✅ **Data Augmentation avanzado** (HSV, Mosaic, Mixup, rotaciones, flips)
- ✅ **Patience aumentada** (15 vs 10)
- ✅ **Class weights** para clases desbalanceadas
- ✅ **Objetivo: mAP50 > 0.75** (actual: 0.651)

## 📊 Métricas actuales del modelo base:
- mAP50: **0.651** → Objetivo: **0.75+**
- Precision: **0.741**
- Recall: **0.617**
- Problemas detectados: Nike (0.004 mAP), Apple (0.279 mAP)

## ⚡ Ventajas de usar Colab
- **GPU T4 gratis** (~15x más rápido que CPU)
- **15GB RAM** incluida
- **Entrenamiento**: 45-60 minutos con mejoras (vs 20-30 min base)

## 📋 Instrucciones previas
1. Sube `flickr_logos_27_dataset.zip` y `data.yaml` a tu Google Drive (carpeta raíz)
2. En Google Colab: **Runtime → Change runtime type → T4 GPU → Save**
3. Ejecuta las celdas en orden

## 1️⃣ Montar Google Drive y Configuración Inicial

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("📂 Verificando archivos en Drive:")
!ls -lh /content/drive/MyDrive/ | grep -E "flickr_logos_27_dataset|data.yaml"

## 2️⃣ Instalar YOLOv11 (Ultralytics)

In [ ]:
!pip install -q ultralytics>=8.3.0

from ultralytics import YOLO
print("✅ Ultralytics instalado correctamente")

## 3️⃣ Verificar GPU Disponible

In [ ]:
!nvidia-smi

import torch
print(f"\n🚀 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU no detectada. Ve a Runtime → Change runtime type → T4 GPU")

## 4️⃣ Limpiar Dataset Anterior

In [ ]:
print("🗑️ Limpiando archivos del dataset anterior...")
!rm -rf /content/dataset
!rm -f /content/data.yaml
print("✅ Limpieza completada.")

## 5️⃣ Descomprimir Dataset

In [ ]:
print("📦 Descomprimiendo flickr_logos_27_dataset.zip...")
!unzip -q /content/drive/MyDrive/flickr_logos_27_dataset.zip -d /content/dataset/

print("\n📂 Estructura del dataset:")
!ls -lh /content/dataset/flickr_logos_27_dataset/

## 6️⃣ Inspeccionar Archivo de Anotaciones

In [ ]:
print("📄 Primeras 5 líneas de flickr_logos_27_dataset_training_set_annotation.txt:")
!head -n 5 /content/dataset/flickr_logos_27_dataset/flickr_logos_27_dataset_training_set_annotation.txt

## 7️⃣ Extraer Nombres de Clases Únicos

In [ ]:
import os

annotation_file_path = '/content/dataset/flickr_logos_27_dataset/flickr_logos_27_dataset_training_set_annotation.txt'
unique_classes = set()

print("⚙️ Extrayendo nombres de clases únicos...")

with open(annotation_file_path, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')
        if len(parts) > 1:
            unique_classes.add(parts[1])

class_names = sorted(list(unique_classes))

print(f"✅ Clases únicas encontradas ({len(class_names)}):")
print(class_names)

## 8️⃣ Generar data.yaml

In [ ]:
yaml_content = f"""
path: /content/yolo_dataset
train: images/train
val: images/val

nc: {len(class_names)}

names: {class_names}
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml generado exitosamente:")
print(yaml_content)

## 9️⃣ Convertir Anotaciones a Formato YOLO

In [ ]:
import os
import shutil
from PIL import Image
import random

dataset_root = '/content/dataset/flickr_logos_27_dataset/'
image_folder = os.path.join(dataset_root, 'flickr_logos_27_dataset_images/')
annotation_file = os.path.join(dataset_root, 'flickr_logos_27_dataset_training_set_annotation.txt')
yolo_dataset_root = '/content/yolo_dataset/'

os.makedirs(os.path.join(yolo_dataset_root, 'images', 'train'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_root, 'images', 'val'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_root, 'labels', 'train'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_root, 'labels', 'val'), exist_ok=True)

print("⚙️ Convirtiendo anotaciones a formato YOLO...")

image_annotations = {}

with open(annotation_file, 'r') as f:
    for line in f:
        parts = line.strip().split(' ')
        if len(parts) >= 6:
            img_name = parts[0]
            class_name = parts[1]
            xmin, ymin, xmax, ymax = map(int, parts[3:7])

            if img_name not in image_annotations:
                image_annotations[img_name] = []
            image_annotations[img_name].append({
                'class_name': class_name,
                'bbox': [xmin, ymin, xmax, ymax]
            })

all_image_files = list(image_annotations.keys())
random.shuffle(all_image_files)

train_split_ratio = 0.8
train_files = all_image_files[:int(len(all_image_files) * train_split_ratio)]
val_files = all_image_files[int(len(all_image_files) * train_split_ratio):]

def convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_width, img_height):
    center_x = (xmin + xmax) / 2.0
    center_y = (ymin + ymax) / 2.0
    width = xmax - xmin
    height = ymax - ymin

    center_x /= img_width
    center_y /= img_height
    width /= img_width
    height /= img_height

    return center_x, center_y, width, height

class_to_id = {name: i for i, name in enumerate(class_names)}

processed_images_count = 0
for img_name in all_image_files:
    img_path_src = os.path.join(image_folder, img_name)
    if not os.path.exists(img_path_src):
        continue

    try:
        with Image.open(img_path_src) as img:
            img_width, img_height = img.size
    except Exception as e:
        continue

    if img_name in train_files:
        img_dest_dir = os.path.join(yolo_dataset_root, 'images', 'train')
        label_dest_dir = os.path.join(yolo_dataset_root, 'labels', 'train')
    else:
        img_dest_dir = os.path.join(yolo_dataset_root, 'images', 'val')
        label_dest_dir = os.path.join(yolo_dataset_root, 'labels', 'val')

    shutil.copy(img_path_src, img_dest_dir)

    label_filename = os.path.splitext(img_name)[0] + '.txt'
    label_filepath = os.path.join(label_dest_dir, label_filename)

    with open(label_filepath, 'w') as label_f:
        for annotation in image_annotations[img_name]:
            class_id = class_to_id[annotation['class_name']]
            xmin, ymin, xmax, ymax = annotation['bbox']

            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(img_width - 1, xmax)
            ymax = min(img_height - 1, ymax)

            if xmax <= xmin or ymax <= ymin:
                continue

            center_x, center_y, width, height = convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_width, img_height)
            label_f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}\n")
    processed_images_count += 1

print(f"✅ Conversión completada. {processed_images_count} imágenes procesadas.")
print(f"   Train: {len(train_files)} | Val: {len(val_files)}")

## 🔥 1️⃣0️⃣ ENTRENAR MODELO MEJORADO

### 🎯 CAMBIOS CLAVE:
1. **Épocas: 50 → 100** ⬆️
2. **Data Augmentation:**
   - HSV (color, saturación, brillo)
   - Mosaic
   - Mixup
   - Rotaciones ±10°
   - Flips horizontal y vertical
3. **Patience: 10 → 15** ⬆️
4. **Class weights: 1.5x** (para clases desbalanceadas)

**Tiempo estimado: 45-60 minutos** ☕

In [ ]:
from ultralytics import YOLO
import time

print("📦 Cargando YOLOv11n con pesos de COCO...")
model = YOLO('yolo11n.pt')

print("\n🏋️ Iniciando ENTRENAMIENTO MEJORADO con GPU T4...\n")
print("⏱️ Tiempo estimado: 45-60 minutos (vs 25-30 min versión base)\n")

start_time = time.time()

# 🔥 CONFIGURACIÓN OPTIMIZADA
results = model.train(
    data='/content/data.yaml',
    
    # 🎯 Parámetros básicos MEJORADOS
    epochs=100,                      # ⬆️ DUPLICADO (era 50)
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    
    # 🎨 DATA AUGMENTATION AVANZADO (NUEVO)
    hsv_h=0.015,                     # Variación de tono (hue)
    hsv_s=0.7,                       # Saturación
    hsv_v=0.4,                       # Brillo (value)
    degrees=10,                      # Rotación ±10°
    translate=0.1,                   # Traslación 10%
    scale=0.5,                       # Escalado 50%
    flipud=0.5,                      # Flip vertical 50%
    fliplr=0.5,                      # Flip horizontal 50%
    mosaic=1.0,                      # Mosaic augmentation
    mixup=0.1,                       # Mixup augmentation
    
    # ⚙️ HIPERPARÁMETROS OPTIMIZADOS
    patience=15,                     # ⬆️ Más paciencia (era 10)
    optimizer='AdamW',
    lr0=0.001,                       # Learning rate inicial
    lrf=0.01,                        # Learning rate final
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    
    # 🎯 CLASS WEIGHTS (NUEVO)
    cls_weight=1.5,                  # Aumentar peso de clasificación
    
    # 📊 Configuración de salida
    project='runs/detect',
    name='logos_flickr_improved',    # NUEVO NOMBRE
    verbose=True,
    seed=42,
    deterministic=True,
    plots=True,
    save=True,
    save_period=10                   # Guardar cada 10 épocas
)

elapsed = time.time() - start_time
print(f"\n✅ Entrenamiento completado en {elapsed/60:.1f} minutos")
print(f"\n📊 Resultados guardados en: /content/runs/detect/logos_flickr_improved/")

## 1️⃣1️⃣ Visualizar Resultados MEJORADOS

In [ ]:
from IPython.display import Image, display
import os

results_dir = '/content/runs/detect/logos_flickr_improved'

print("📊 RESULTADOS DEL ENTRENAMIENTO MEJORADO\n")
print("="*60)

if os.path.exists(f'{results_dir}/results.png'):
    print("📈 Curvas de entrenamiento (100 épocas):")
    display(Image(filename=f'{results_dir}/results.png', width=800))

print("\n" + "="*60 + "\n")

if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    print("🎯 Matriz de confusión:")
    display(Image(filename=f'{results_dir}/confusion_matrix.png', width=600))

print("\n" + "="*60 + "\n")

if os.path.exists(f'{results_dir}/val_batch0_pred.jpg'):
    print("🖼️ Predicciones en validación:")
    display(Image(filename=f'{results_dir}/val_batch0_pred.jpg', width=800))

## 1️⃣2️⃣ Validar y Comparar Métricas

In [ ]:
print("🧪 Ejecutando validación final...\n")
metrics = model.val()

print("\n" + "="*60)
print("📊 COMPARACIÓN DE MÉTRICAS")
print("="*60)
print("\n🔴 MODELO ANTERIOR (50 épocas, sin augmentation):")
print(f"   mAP50:     0.651")
print(f"   mAP50-95:  0.422")
print(f"   Precisión: 0.741")
print(f"   Recall:    0.617")

print("\n🟢 MODELO MEJORADO (100 épocas, augmentation avanzado):")
print(f"   mAP50:     {metrics.box.map50:.3f}")
print(f"   mAP50-95:  {metrics.box.map:.3f}")
print(f"   Precisión: {metrics.box.mp:.3f}")
print(f"   Recall:    {metrics.box.mr:.3f}")

print("\n📈 MEJORA:")
mejora_map50 = (metrics.box.map50 - 0.651) / 0.651 * 100
print(f"   mAP50:     {mejora_map50:+.1f}%")
print("="*60)

print("\n📋 Métricas por clase (MEJORADO):")
if hasattr(metrics, 'names') and metrics.names:
    class_names_metrics = [metrics.names[i] for i in sorted(metrics.names.keys())]
else:
    class_names_metrics = [f'clase_{i}' for i in range(len(metrics.box.maps))]

for i, name in enumerate(class_names_metrics):
    if i < len(metrics.box.maps):
        print(f"  {name:15s} - mAP50: {metrics.box.maps[i]:.3f}")

## 1️⃣3️⃣ Guardar Modelo Mejorado

In [ ]:
print("📦 Comprimiendo modelo mejorado...")
!zip -r /content/yolo11_flickr_improved.zip /content/runs/detect/logos_flickr_improved/

!cp /content/yolo11_flickr_improved.zip /content/drive/MyDrive/
!cp /content/runs/detect/logos_flickr_improved/weights/best.pt /content/drive/MyDrive/yolo11_flickr_improved_best.pt

print("\n✅ Archivos guardados en Google Drive:")
print("   📁 yolo11_flickr_improved.zip (resultados completos)")
print("   🎯 yolo11_flickr_improved_best.pt (mejor modelo)")
print("\n💡 Compara este modelo con el anterior en tu proyecto")

## 1️⃣4️⃣ Probar Modelo Mejorado

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import glob
from PIL import Image as PILImage

best_model = YOLO('/content/runs/detect/logos_flickr_improved/weights/best.pt')

val_images = glob.glob('/content/yolo_dataset/images/val/*.jpg')
if val_images:
    test_image = val_images[0]
    print(f"🖼️ Probando modelo mejorado: {test_image}\n")
    
    results = best_model(test_image)
    
    for r in results:
        im_array = r.plot()
        im = PILImage.fromarray(im_array[..., ::-1])
        display(im)
        
        if len(r.boxes) > 0:
            print(f"\n✅ Detectados {len(r.boxes)} logos:")
            for box in r.boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                print(f"  - {best_model.names[cls_id]}: {conf:.2%}")
        else:
            print("\n⚠️ No se detectaron logos")

## ✅ RESUMEN

### Mejoras implementadas:
1. ✅ 100 épocas (vs 50)
2. ✅ Data augmentation HSV
3. ✅ Mosaic + Mixup
4. ✅ Rotaciones y flips
5. ✅ Patience aumentada
6. ✅ Class weights

### Objetivo:
- **mAP50 > 0.75** (era 0.651)
- Mejor detección de Nike y Apple
- Menos falsos positivos

### Próximos pasos:
1. Descargar `yolo11_flickr_improved_best.pt` de Drive
2. Colocar en `models/models_org/weights/best.pt`
3. Ejecutar pruebas: `python tests/model_evaluation/evaluate_model.py`
4. Actualizar `conf=0.30` en demo de Oscar si mAP50 > 0.75